# EduPro — Predictive Modeling for Course Demand & Revenue Forecasting

**Primary ML target:** Next-month course enrollment demand  
**Derived business output:** Revenue forecast = predicted enrollments × course price

## Workflow
1. Data Loading & Understanding
2. Data Cleaning & Validation
3. Exploratory Data Analysis (EDA)
4. Data Integration
5. Feature Engineering
6. Target Construction
7. Time-based Train/Test Split
8. Preprocessing
9. Baseline & Advanced Models
10. Evaluation
11. Feature Importance
12. Revenue Forecasting
13. Category-Level Forecasting
14. Export for Streamlit / Research Paper

> Keep this as the master ML notebook so every result remains reproducible.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

print("Libraries loaded.")

## 1. Load Raw Data

Upload the four CSV files to Colab/Jupyter and update paths if required.

In [ ]:
TRANSACTIONS_PATH = "Transactions.csv"
COURSES_PATH = "Courses.csv"
TEACHERS_PATH = "Teachers.csv"
USERS_PATH = "Users.csv"

transactions = pd.read_csv(TRANSACTIONS_PATH)
courses = pd.read_csv(COURSES_PATH)
teachers = pd.read_csv(TEACHERS_PATH)
users = pd.read_csv(USERS_PATH)

for name, df in {
    "Transactions": transactions,
    "Courses": courses,
    "Teachers": teachers,
    "Users": users
}.items():
    print(f"{name}: {df.shape}")

In [ ]:
display(transactions.head())
display(courses.head())
display(teachers.head())
display(users.head())

## 2. Data Understanding & Validation

In [ ]:
datasets = {
    "Transactions": transactions,
    "Courses": courses,
    "Teachers": teachers,
    "Users": users
}

for name, df in datasets.items():
    print("\n" + "="*60)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nMissing values:")
    display(df.isna().sum().to_frame("Missing"))

## 3. Data Cleaning

In [ ]:
transactions["TransactionDate"] = pd.to_datetime(
    transactions["TransactionDate"],
    dayfirst=True,
    errors="coerce"
)

assert transactions["TransactionDate"].notna().all(), "Invalid dates found."

transactions = transactions.drop_duplicates().copy()
courses = courses.drop_duplicates().copy()
teachers = teachers.drop_duplicates().copy()
users = users.drop_duplicates().copy()

print("Cleaning complete.")

In [ ]:
print("Invalid CourseIDs:",
      (~transactions["CourseID"].isin(courses["CourseID"])).sum())
print("Invalid UserIDs:",
      (~transactions["UserID"].isin(users["UserID"])).sum())
print("Invalid TeacherIDs:",
      (~transactions["TeacherID"].isin(teachers["TeacherID"])).sum())

## 4. Exploratory Data Analysis (EDA)

Questions:
- How many courses, learners and transactions exist?
- Which categories have the most demand?
- Which categories generate the most revenue?
- How does price relate to enrollment?
- Does rating relate to demand?
- How does demand change over time?

In [ ]:
course_sales = transactions.groupby("CourseID").agg(
    Enrollments=("TransactionID", "count"),
    Revenue=("Amount", "sum"),
    ActiveDays=("TransactionDate", "nunique")
).reset_index()

course_eda = courses.merge(course_sales, on="CourseID", how="left")
course_eda[["Enrollments", "Revenue", "ActiveDays"]] = (
    course_eda[["Enrollments", "Revenue", "ActiveDays"]].fillna(0)
)
course_eda["RevenuePerEnrollment"] = np.where(
    course_eda["Enrollments"] > 0,
    course_eda["Revenue"] / course_eda["Enrollments"],
    0
)

print("Transactions:", len(transactions))
print("Courses:", courses["CourseID"].nunique())
print("Users:", users["UserID"].nunique())
print("Total revenue:", round(transactions["Amount"].sum(), 2))
print("Average enrollments/course:", round(course_eda["Enrollments"].mean(), 2))

In [ ]:
category_eda = course_eda.groupby("CourseCategory").agg(
    Courses=("CourseID", "count"),
    Enrollments=("Enrollments", "sum"),
    Revenue=("Revenue", "sum"),
    AvgPrice=("CoursePrice", "mean"),
    AvgRating=("CourseRating", "mean")
).reset_index()

category_eda["RevenuePerEnrollment"] = (
    category_eda["Revenue"] / category_eda["Enrollments"]
)

display(category_eda.sort_values("Enrollments", ascending=False))

In [ ]:
transactions["Month"] = (
    transactions["TransactionDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_eda = transactions.groupby("Month").agg(
    Enrollments=("TransactionID", "count"),
    Revenue=("Amount", "sum")
).reset_index()

display(monthly_eda)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(course_eda["CoursePrice"], course_eda["Enrollments"])
plt.xlabel("Course Price")
plt.ylabel("Enrollment Count")
plt.title("Course Price vs Enrollment")
plt.tight_layout()
plt.show()

display(
    course_eda[
        ["CoursePrice", "CourseDuration", "CourseRating", "Enrollments"]
    ].corr()["Enrollments"].sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(monthly_eda["Month"], monthly_eda["Enrollments"], marker="o")
plt.xlabel("Month")
plt.ylabel("Enrollments")
plt.title("Monthly Enrollment Trend — 2025")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Data Integration

Teacher IDs are not treated as a definitive one-course/one-teacher mapping because the transaction data can associate multiple teacher IDs with a course. We therefore create aggregated instructor features.

In [ ]:
teacher_transaction = (
    transactions
    .merge(courses[["CourseID", "CourseCategory"]], on="CourseID", how="left")
    .merge(
        teachers[
            ["TeacherID", "Expertise", "YearsOfExperience", "TeacherRating"]
        ],
        on="TeacherID",
        how="left"
    )
)

teacher_transaction["ExpertiseMatch"] = (
    teacher_transaction["CourseCategory"] ==
    teacher_transaction["Expertise"]
).astype(int)

teacher_features = teacher_transaction.groupby("CourseID").agg(
    AvgTeacherExperience=("YearsOfExperience", "mean"),
    AvgTeacherRating=("TeacherRating", "mean"),
    TeacherCount=("TeacherID", "nunique"),
    ExpertiseMatchRate=("ExpertiseMatch", "mean")
).reset_index()

display(teacher_features.head())

In [ ]:
user_transaction = transactions.merge(
    users[["UserID", "Age", "Gender"]],
    on="UserID",
    how="left"
)

user_features = user_transaction.groupby("CourseID").agg(
    AvgLearnerAge=("Age", "mean"),
    UniqueLearners=("UserID", "nunique"),
    FemaleLearnerShare=("Gender", lambda x: (x == "Female").mean())
).reset_index()

display(user_features.head())

## 6. Build Monthly Course Panel

A course-level dataset has only 60 observations. A monthly panel gives **60 courses × 12 months = 720 course-month observations**, which is more suitable for forecasting.

In [ ]:
monthly_course = (
    transactions.groupby(["CourseID", "Month"])
    .agg(
        EnrollmentCount=("TransactionID", "count"),
        Revenue=("Amount", "sum")
    )
    .reset_index()
)

all_courses = courses["CourseID"].unique()
all_months = pd.date_range(
    transactions["Month"].min(),
    transactions["Month"].max(),
    freq="MS"
)

grid = pd.MultiIndex.from_product(
    [all_courses, all_months],
    names=["CourseID", "Month"]
).to_frame(index=False)

monthly_course = (
    grid.merge(monthly_course, on=["CourseID", "Month"], how="left")
    .fillna({"EnrollmentCount": 0, "Revenue": 0})
)

print("Panel shape:", monthly_course.shape)
display(monthly_course.head())

## 7. Feature Engineering

In [ ]:
model_df = (
    monthly_course
    .merge(courses, on="CourseID", how="left")
    .merge(teacher_features, on="CourseID", how="left")
    .merge(user_features, on="CourseID", how="left")
)

model_df["IsFreeCourse"] = (model_df["CoursePrice"] == 0).astype(int)
model_df["MonthNumber"] = model_df["Month"].dt.month
model_df["Quarter"] = model_df["Month"].dt.quarter

model_df = model_df.sort_values(["CourseID", "Month"]).reset_index(drop=True)
grouped = model_df.groupby("CourseID")

model_df["Lag1_Enrollments"] = grouped["EnrollmentCount"].shift(1)
model_df["Lag3_Enrollments"] = grouped["EnrollmentCount"].shift(3)
model_df["Rolling3_Enrollments"] = grouped["EnrollmentCount"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
model_df["Lag1_Revenue"] = grouped["Revenue"].shift(1)
model_df["Rolling3_Revenue"] = grouped["Revenue"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

display(model_df.head(15))

## 8. Target Construction

**Primary target:** next-month enrollment demand.

The target is shifted so features from month `t` predict enrollment in month `t+1`.

In [ ]:
model_df["Target_NextMonthEnrollments"] = (
    grouped["EnrollmentCount"].shift(-1)
)

model_df["Target_NextMonthRevenue"] = (
    model_df["Target_NextMonthEnrollments"] *
    model_df["CoursePrice"]
)

model_df = model_df.dropna(
    subset=["Target_NextMonthEnrollments"]
).copy()

print("Modeling rows:", len(model_df))
display(model_df.head())

## 9. Time-Based Train/Test Split

Do **not** randomly split this forecasting dataset. Train on earlier months and test on later months to avoid future-information leakage.

In [ ]:
cutoff = pd.Timestamp("2025-09-01")

train_df = model_df[model_df["Month"] < cutoff].copy()
test_df = model_df[model_df["Month"] >= cutoff].copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)
print("Train period:", train_df["Month"].min(), "to", train_df["Month"].max())
print("Test period :", test_df["Month"].min(), "to", test_df["Month"].max())

## 10. Preprocessing

In [ ]:
target = "Target_NextMonthEnrollments"

feature_cols = [
    "CourseCategory", "CourseType", "CourseLevel",
    "CoursePrice", "IsFreeCourse", "CourseDuration", "CourseRating",
    "AvgTeacherExperience", "AvgTeacherRating", "TeacherCount",
    "ExpertiseMatchRate", "AvgLearnerAge", "UniqueLearners",
    "FemaleLearnerShare", "MonthNumber", "Quarter",
    "Lag1_Enrollments", "Lag3_Enrollments",
    "Rolling3_Enrollments", "Lag1_Revenue", "Rolling3_Revenue"
]

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test = test_df[feature_cols]
y_test = test_df[target]

categorical_features = ["CourseCategory", "CourseType", "CourseLevel"]
numeric_features = [x for x in feature_cols if x not in categorical_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print("Preprocessor ready.")

## 11. Train Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3
    )
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

## 12. Model Evaluation

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
test_predictions = best_model.predict(X_test)

evaluation_df = test_df[
    ["CourseID", "Month", "CourseCategory", "CoursePrice",
     "Target_NextMonthEnrollments"]
].copy()

evaluation_df["PredictedEnrollments"] = test_predictions
evaluation_df["AbsoluteError"] = (
    evaluation_df["Target_NextMonthEnrollments"] -
    evaluation_df["PredictedEnrollments"]
).abs()

print("Model selected by lowest RMSE:", best_model_name)
display(evaluation_df.head(20))

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_predictions)
plt.xlabel("Actual Next-Month Enrollments")
plt.ylabel("Predicted Next-Month Enrollments")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()

## 13. Feature Importance

In [ ]:
if best_model_name in ["Random Forest", "Gradient Boosting"]:
    fitted_preprocessor = best_model.named_steps["preprocessor"]
    estimator = best_model.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": estimator.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(20))

    plt.figure(figsize=(9, 6))
    top = importance_df.head(15).sort_values("Importance")
    plt.barh(top["Feature"], top["Importance"])
    plt.xlabel("Importance")
    plt.title("Top Feature Importances")
    plt.tight_layout()
    plt.show()
else:
    print("Native feature importance is available after selecting a tree-based model.")

## 14. Revenue Forecasting

The dataset shows that transaction amount equals course price, so:

**Forecast Revenue = Forecast Enrollment × Course Price**

We therefore derive revenue from the enrollment model rather than creating a redundant revenue target.

In [ ]:
revenue_forecast = evaluation_df.copy()

revenue_forecast["ActualRevenue"] = (
    revenue_forecast["Target_NextMonthEnrollments"] *
    revenue_forecast["CoursePrice"]
)
revenue_forecast["PredictedRevenue"] = (
    revenue_forecast["PredictedEnrollments"] *
    revenue_forecast["CoursePrice"]
)

display(revenue_forecast.head(20))

## 15. Category-Level Forecast

In [ ]:
category_forecast = (
    revenue_forecast.groupby("CourseCategory")
    .agg(
        ActualEnrollments=("Target_NextMonthEnrollments", "sum"),
        PredictedEnrollments=("PredictedEnrollments", "sum"),
        ActualRevenue=("ActualRevenue", "sum"),
        PredictedRevenue=("PredictedRevenue", "sum")
    )
    .reset_index()
)

display(category_forecast.sort_values("PredictedRevenue", ascending=False))

## 16. Export Artifacts

In [ ]:
model_df.to_csv("EduPro_modeling_dataset.csv", index=False)
results_df.to_csv("EduPro_model_results.csv", index=False)
evaluation_df.to_csv("EduPro_course_forecasts.csv", index=False)
category_forecast.to_csv("EduPro_category_forecasts.csv", index=False)

if "importance_df" in globals():
    importance_df.to_csv("EduPro_feature_importance.csv", index=False)

print("All modeling artifacts exported.")

## PHASE 4 — ML Model Development & Evaluation

We now train five regression models on the leakage-safe Phase 3 dataset:

1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Random Forest Regressor
5. Gradient Boosting Regressor

The test set contains later months than the training set, preserving the forecasting setup.

In [ ]:
# Time-based train/test split
cutoff = pd.Timestamp("2025-09-01")
train_df = model_df[model_df["Month"] < cutoff].copy()
test_df = model_df[model_df["Month"] >= cutoff].copy()

target = "Target_NextMonthEnrollments"

feature_cols = [
    "CourseCategory","CourseType","CourseLevel",
    "CoursePrice","LogCoursePrice","IsFreeCourse",
    "CourseDuration","CourseRating","DurationBucket",
    "MonthNumber","Quarter",
    "Lag1_Enrollments","Lag2_Enrollments","Lag3_Enrollments",
    "Rolling3_Enrollments","Lag1_Revenue","Rolling3_Revenue",
    "EnrollmentMomentum"
]

categorical_features = [
    "CourseCategory","CourseType","CourseLevel","DurationBucket"
]
numeric_features = [
    x for x in feature_cols if x not in categorical_features
]

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test = test_df[feature_cols]
y_test = test_df[target]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print("Train:", X_train.shape)
print("Test :", X_test.shape)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=20000, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=500, random_state=42, n_jobs=-1, min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=2,
        min_samples_leaf=3, random_state=42
    )
}

results = []
fitted_models = {}
predictions = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

    fitted_models[name] = pipe
    predictions[name] = pred

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

### Phase 4 model comparison

The model should be selected from the measured test results rather than assuming that a more complex model is better.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_predictions = predictions[best_model_name]

print("Best model by lowest RMSE:", best_model_name)
display(results_df)

In [ ]:
# Actual vs predicted
plt.figure(figsize=(7,6))
plt.scatter(y_test, best_predictions)
mn = min(y_test.min(), best_predictions.min())
mx = max(y_test.max(), best_predictions.max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Actual Next-Month Enrollments")
plt.ylabel("Predicted Next-Month Enrollments")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Native feature importance for tree models
if best_model_name in ["Random Forest", "Gradient Boosting"]:
    pp = best_model.named_steps["preprocessor"]
    estimator = best_model.named_steps["model"]

    feature_names = pp.get_feature_names_out()
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": estimator.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(20))
else:
    importance_df = pd.DataFrame()
    print("Selected model is not a tree model; use coefficients for linear interpretation.")

In [ ]:
# Build course-level test forecasts
evaluation_df = test_df[
    ["CourseID","Month","CourseCategory","CoursePrice",
     "Target_NextMonthEnrollments"]
].copy()

evaluation_df["PredictedEnrollments"] = best_predictions
evaluation_df["AbsoluteError"] = (
    evaluation_df["Target_NextMonthEnrollments"] -
    evaluation_df["PredictedEnrollments"]
).abs()

evaluation_df["ActualRevenue"] = (
    evaluation_df["Target_NextMonthEnrollments"] *
    evaluation_df["CoursePrice"]
)
evaluation_df["PredictedRevenue"] = (
    evaluation_df["PredictedEnrollments"] *
    evaluation_df["CoursePrice"]
)

display(evaluation_df.head(20))

In [ ]:
# Category-level forecast
category_forecast = (
    evaluation_df.groupby("CourseCategory")
    .agg(
        ActualEnrollments=("Target_NextMonthEnrollments","sum"),
        PredictedEnrollments=("PredictedEnrollments","sum"),
        ActualRevenue=("ActualRevenue","sum"),
        PredictedRevenue=("PredictedRevenue","sum")
    )
    .reset_index()
)

display(category_forecast)

In [ ]:
# Export Phase 4 artifacts
results_df.to_csv("EduPro_Phase4_Model_Results.csv", index=False)
evaluation_df.to_csv("EduPro_Phase4_Best_Model_Forecasts.csv", index=False)
category_forecast.to_csv("EduPro_Phase4_Category_Forecasts.csv", index=False)

if not importance_df.empty:
    importance_df.to_csv("EduPro_Phase4_Feature_Importance.csv", index=False)

print("Phase 4 artifacts exported.")

# 17. Final Insights — Complete After Modeling

Do not write conclusions before seeing the actual model results.

### Demand
- Which features are most important?
- How much does historical demand matter?
- What is the observed role of price and rating?

### Revenue
- Which categories have high demand?
- Which categories monetize demand?
- Which courses combine demand and price effectively?

### Model
- MAE:
- RMSE:
- R²:
- Largest forecast errors:
- Final model:

### Business use
- Course launch planning
- Pricing analysis
- Instructor/resource allocation
- Category planning
- Free-to-paid strategy

# 18. Completion Checklist

- [x] Data loading
- [x] Validation
- [x] Cleaning
- [x] EDA
- [x] Data integration
- [x] Monthly panel
- [x] Feature engineering
- [x] Target construction
- [x] Time split
- [x] Baseline + advanced models
- [x] Evaluation
- [x] Feature importance
- [x] Revenue derivation
- [x] Category forecasting

## Remaining
- [ ] Hyperparameter tuning
- [ ] Time-series cross-validation
- [ ] Final model selection
- [ ] Streamlit dashboard
- [ ] Research paper
- [ ] Executive summary

## PHASE 3 — Final Leakage-Safe Feature Engineering

This section is the authoritative Phase 3 implementation.

**Important:** full-period learner/teacher transaction aggregates are not used as forecasting features because they can contain information from future months. We use static course attributes plus historical course behavior available before each prediction month.

In [ ]:
# Complete course × month panel
monthly_course = (
    transactions.groupby(["CourseID", "Month"])
    .agg(EnrollmentCount=("TransactionID", "count"),
         Revenue=("Amount", "sum"))
    .reset_index()
)

all_months = pd.date_range(
    transactions["Month"].min(),
    transactions["Month"].max(),
    freq="MS"
)

grid = pd.MultiIndex.from_product(
    [courses["CourseID"].unique(), all_months],
    names=["CourseID", "Month"]
).to_frame(index=False)

monthly_course = (
    grid.merge(monthly_course, on=["CourseID", "Month"], how="left")
    .fillna({"EnrollmentCount": 0, "Revenue": 0})
)

print("Complete panel:", monthly_course.shape)
display(monthly_course.head())

In [ ]:
# Static + time features
model_df = monthly_course.merge(courses, on="CourseID", how="left")

model_df["IsFreeCourse"] = (model_df["CoursePrice"] == 0).astype(int)
model_df["LogCoursePrice"] = np.log1p(model_df["CoursePrice"])
model_df["MonthNumber"] = model_df["Month"].dt.month
model_df["Quarter"] = model_df["Month"].dt.quarter

model_df["DurationBucket"] = pd.cut(
    model_df["CourseDuration"],
    bins=[-np.inf, 10, 30, 60, np.inf],
    labels=["Short", "Medium", "Long", "Very Long"]
)

model_df = model_df.sort_values(["CourseID", "Month"]).reset_index(drop=True)
g = model_df.groupby("CourseID")

# Historical features: every one uses only previous months
model_df["Lag1_Enrollments"] = g["EnrollmentCount"].shift(1)
model_df["Lag2_Enrollments"] = g["EnrollmentCount"].shift(2)
model_df["Lag3_Enrollments"] = g["EnrollmentCount"].shift(3)

model_df["Rolling3_Enrollments"] = g["EnrollmentCount"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

model_df["Lag1_Revenue"] = g["Revenue"].shift(1)

model_df["Rolling3_Revenue"] = g["Revenue"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

model_df["EnrollmentMomentum"] = (
    model_df["Lag1_Enrollments"] - model_df["Lag3_Enrollments"]
)

display(model_df.head(12))

In [ ]:
# Primary target: next-month enrollment
model_df["Target_NextMonthEnrollments"] = (
    g["EnrollmentCount"].shift(-1)
)

# Revenue is derived from predicted enrollment × course price
model_df["Target_NextMonthRevenue"] = (
    model_df["Target_NextMonthEnrollments"] *
    model_df["CoursePrice"]
)

# Require 3 months of prior history and a future target
model_df = model_df.dropna(
    subset=["Lag3_Enrollments", "Target_NextMonthEnrollments"]
).copy()

print("ML-ready rows:", len(model_df))
print("Courses:", model_df["CourseID"].nunique())
print("Months:", model_df["Month"].nunique())

### Phase 3 validation

In [ ]:
# Leakage / structural validation
assert model_df["Target_NextMonthEnrollments"].notna().all()
assert model_df["Lag3_Enrollments"].notna().all()
assert model_df["CourseID"].nunique() == 60

feature_cols = [
    "CourseCategory", "CourseType", "CourseLevel",
    "CoursePrice", "LogCoursePrice", "IsFreeCourse",
    "CourseDuration", "CourseRating", "DurationBucket",
    "MonthNumber", "Quarter",
    "Lag1_Enrollments", "Lag2_Enrollments", "Lag3_Enrollments",
    "Rolling3_Enrollments", "Lag1_Revenue", "Rolling3_Revenue",
    "EnrollmentMomentum"
]

print("Feature count:", len(feature_cols))
print("Target:", "Target_NextMonthEnrollments")
print("Phase 3 validation PASSED.")
display(
    model_df[
        ["CourseID","Month","Lag1_Enrollments",
         "Lag3_Enrollments","Rolling3_Enrollments",
         "Target_NextMonthEnrollments"]
    ].head(12)
)